# JOINTS 2026 Blend PoC Notebook

Objective: Membandingkan LightGBM dengan model dari jenis lain (Linear Regression, SVM, LSTM, dan Logistic Regression untuk target nol), pada tiga lipatan validasi waktu (Juli, Agustus, September 2025) dengan MASE sebagai metrik evaluasi.

# Sinkronisasi versi modules dan libraries

In [ ]:
import sys
print(sys.version)
%pip install -q numpy==2.4.4 pandas==2.3.3 scipy==1.17.1 scikit-learn==1.8.0 lightgbm==4.6.0 torch==2.11.0 tqdm==4.67.3

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\leouw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Import modules dan libraries

In [ ]:
import os, glob, random, re, time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.svm import LinearSVR
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Import datasets

In [ ]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

### External datasets
Kalender publik yang tidak ada pada file holidays bawaan: hari cuti bersama, tanggal Ramadan, libur sekolah, dan libur nasional tambahan, ditambah tabel profil film dan chart box office nasional. Semuanya dipublikasikan pada atau sebelum 2025-09-30, sehingga tidak ada data dari periode test yang dipakai.

In [ ]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

# Aggregate Dataset
Mengubah catatan harian menjadi satu baris per pasangan film dan bioskop untuk setiap hari target, dengan format yang sama untuk train dan test.

### Penentuan D1
D1 adalah hari pertama film tayang di setidaknya setengah dari jumlah bioskop pada rilis terluasnya, sehingga preview dan tayang terbatas tidak menentukan D1. Film yang sudah tayang sebelum periode train dimulai, atau yang D10-nya jatuh setelah periode train berakhir, dikeluarkan karena jendela D1-D10 nya tidak lengkap.

In [ ]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

### Pasangan film dan bioskop
Membangun jendela D1-D10 untuk setiap pasangan dengan tiket dan jumlah show per hari. D1-D3 menjadi input dan D4-D10 menjadi target; pasangan test hanya memiliki D1-D3. Pasangan train tanpa penjualan pada D3 dibuang, sama seperti definisi pasangan test.

In [ ]:
DAILY_COLS = {"total_ticket": "t", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

### Format harian
Satu baris per pasangan dan hari target (horizon 4 sampai 10), sehingga satu model memprediksi semua horizon dan horizon menjadi fitur.

In [ ]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

### Penggabungan sumber data
Menambahkan kota tiap bioskop, tipe hari libur, batas harga tiket, dan metadata film. Akhiran 2D, 3D, dan IMAX dihapus dari judul sebelum penggabungan metadata, karena semua versi format dari satu film memiliki informasi film yang sama.

In [ ]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

def join_sources(df):
    n = len(df)
    df = df.copy()
    df["city_name"] = df["cinema_ids"].map(CINEMA_CITY)
    df = df.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
    df["price_day"] = df["day_tipe"].map(PRICE_DAY)
    df = df.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
    df["base_title"] = df["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
    df = df.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")
    assert len(df) == n
    return df

full = join_sources(full)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

### Penggabungan external datasets
Menambahkan cuti bersama dan libur sekolah ke setiap baris berdasarkan tanggal.

In [ ]:
def join_external(df):
    n = len(df)
    df = df.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
    df = df.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")
    assert len(df) == n
    return df

full = join_external(full)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "school_break"]].count()

In [ ]:
full.head()

# Preprocessing

### Hapus hari yang tidak lengkap
Hari ketika train.csv memiliki baris jauh lebih sedikit dari hari-hari di sekitarnya adalah rekaman parsial. Baris train yang hari targetnya atau jendela D1-D3 nya menyentuh hari seperti itu dihapus, karena tiketnya tercatat terlalu rendah.

In [ ]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

def drop_gap_rows(df):
    d13_on_gap = np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(GAP_DAYS) for k in range(3)], axis=0)
    broken = (df["is_train"] == 1) & (df["date_show"].isin(GAP_DAYS) | d13_on_gap)
    return df[~broken].reset_index(drop=True)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
n_before = len(full)
full = drop_gap_rows(full)
print(f"dropped train rows: {n_before - len(full)} | full: {full.shape}")

### Flag kalender
Hari libur membuat lebih banyak orang datang ke bioskop. File holidays hanya menandai libur nasional, sedangkan cuti bersama dan libur sekolah berasal dari kalender eksternal, sehingga masing-masing diberi flag sendiri.

In [ ]:
def add_calendar_flags(df):
    df = df.copy()
    df["is_holiday"] = (df["holiday_tipe"] == "holiday").astype(int)
    df["is_cuti_bersama"] = df["cuti_bersama_name"].notna().astype(int)
    df["is_school_holiday"] = df["school_break"].notna().astype(int)
    return df

full = add_calendar_flags(full)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

### Kolom genre
Genre mengubah seberapa cepat penjualan film turun: film keluarga dan animasi bertahan lebih lama, sedangkan film horor cepat turun. Setiap genre menjadi satu kolom 0 atau 1, dan hanya genre yang muncul pada setidaknya 2% baris train yang dipakai, karena genre yang lebih jarang terlalu sedikit barisnya untuk dipelajari model.

In [ ]:
def add_genres(df, genre_cols=None):
    dummies = df["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
    dummies.columns = dummies.columns.str.replace(" ", "_")
    if genre_cols is not None:
        dummies = dummies.reindex(columns=genre_cols, fill_value=0)
    return pd.concat([df, dummies], axis=1), list(dummies.columns)

full, GENRE_COLS = add_genres(full)
COMMON_GENRES = [g for g in GENRE_COLS if full.loc[full["is_train"] == 1, g].mean() >= 0.02]
print(f"genres: {len(GENRE_COLS)} | used as features: {len(COMMON_GENRES)}")
full.loc[full["is_train"] == 1, GENRE_COLS].mean().sort_values(ascending=False).round(3).head(10)

### Kolom kategori
city_name, day_tipe, dan age_rating dideklarasikan sebagai kategori dengan level yang sama pada train dan test, sehingga tree memperlakukannya sebagai kelompok dan bukan angka berurutan, dan kodenya sama antara train dan test.

In [ ]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
CAT_LEVELS = {col: full[col].astype("category").cat.categories for col in CAT_COLS}

def set_categories(df):
    df = df.copy()
    for col in CAT_COLS:
        df[col] = pd.Categorical(df[col], categories=CAT_LEVELS[col])
    return df

full = set_categories(full)
full[CAT_COLS].nunique()

### Hapus kolom tanpa sinyal
Kolom kredit (producer, director, writer, casts) hampir unik untuk setiap film sehingga model hanya bisa menghafalnya. Kolom lain yang dihapus hanya dipakai untuk penggabungan atau mengulang informasi yang sudah ter-encode.

In [ ]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]

def drop_unused(df):
    return df.drop(columns=DROP_COLS)

full = drop_unused(full)
print(f"full: {full.shape}")
print(list(full.columns))

### Target
target_ratio adalah tiket pada hari target dibagi scale, yaitu rata-rata D1-D3 (minimal 1). Metrik kompetisi adalah selisih absolut dibagi scale yang sama, sehingga mean absolute error dari target_ratio adalah metriknya sendiri, dan pasangan dengan ukuran yang sangat berbeda menjadi sebanding.

In [ ]:
def add_target(df):
    df = df.copy()
    df["scale"] = df[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
    df["target_ratio"] = df["total_ticket"] / df["scale"]
    return df

full = add_target(full)
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

# Feature Engineering

### Blok hari libur
Penonton tetap banyak selama rangkaian hari libur berturut-turut dan turun ketika rangkaian itu berakhir, sehingga long weekend berperilaku berbeda dari akhir pekan biasa.
- off_block_len: panjang rangkaian hari libur tanpa putus (akhir pekan, libur nasional, cuti bersama) yang memuat hari target, bernilai 0 untuk hari kerja.

In [ ]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]

def add_off_block(df):
    return df.assign(off_block_len=df["date_show"].map(BLOCK_LEN))

full = add_off_block(full)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

### Fitur pasangan
Perilaku pasangan itu sendiri pada D1-D3, informasi terkuat untuk hari-hari berikutnya:
- trend: tiket D3 relatif terhadap D1, apakah permintaan pasangan naik atau turun.
- t3_share: porsi tiket D1-D3 yang terjual pada D3, menandakan puncak yang terlambat atau mulai tayang terlambat.
- show_mean: rata-rata jumlah show per hari, kapasitas yang diberikan bioskop untuk film tersebut.
- tickets_per_show3: tiket per show pada D3, permintaan yang tidak bergantung pada jumlah show yang dijadwalkan.
- active_days: jumlah hari dengan penjualan pada D1-D3, pasangan yang mulai terlambat memiliki lebih sedikit.

In [ ]:
D13 = ["t1", "t2", "t3"]

def add_pair_features(df):
    df = df.copy()
    df["trend"] = (df["t3"] + 1) / (df["t1"] + 1)
    df["t3_share"] = df["t3"] / df[D13].sum(axis=1)
    df["show_mean"] = df[["show1", "show2", "show3"]].mean(axis=1)
    df["tickets_per_show3"] = df["t3"] / df["show3"].replace(0, np.nan)
    df["active_days"] = (df[D13] > 0).sum(axis=1)
    return df

full = add_pair_features(full)
full[["trend", "t3_share", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

### Fitur film
Film yang sama di semua bioskop, dijumlahkan dari D1-D3. Film diprediksi lebih baik dari gambaran nasionalnya daripada dari satu bioskop saja.
- mv_t1, mv_t2, mv_t3: tiket nasional film pada masing-masing D1-D3, kekuatan film.
- mv_clusters: jumlah bioskop yang memiliki penjualan pada D3, luasnya rilis.
- mv_trend: D3 nasional relatif terhadap D1, tren yang lebih stabil daripada tren satu pasangan.
- pair_share: porsi tiket D1-D3 nasional film yang dijual oleh bioskop ini, seberapa penting bioskop tersebut bagi film.

In [ ]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]

def add_film_features(df, window):
    df = df.copy()
    feats = film_features(window)
    for col in FILM_COLS:
        df[col] = df["movie_title"].map(feats[col]).values
    df["pair_share"] = df[D13].sum(axis=1) / df[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
    return df

is_tr = full["is_train"] == 1
full = pd.concat([add_film_features(full[is_tr], train_window), add_film_features(full[~is_tr], test_window)]).sort_index()
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

### Fitur bioskop
Ukuran dan program tayang yang biasa dari bioskop, dihitung dari jendela D1-D3 semua film tanpa informasi target:
- cl_scale: rata-rata scale D1-D3 film yang tayang, bioskop besar menurun berbeda dari bioskop kecil.
- cl_movies: jumlah film berbeda yang tayang, berapa banyak film yang bersaing untuk layar yang sama.

In [ ]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)

def add_cinema_features(df):
    return df.join(cinema_feats, on="cinema_ids")

full = add_cinema_features(full)
cinema_feats.describe().round(2)

### Fitur hari dan harga
- dow: hari dalam seminggu dari hari target, akhir pekan lebih ramai.
- d1_dow: hari dalam seminggu dari D1, posisi akhir pekan di dalam D1-D10 mengubah bentuk penurunan.
- price_ratio: harga tiket pada hari target relatif terhadap rata-rata harga D1-D3, hari yang lebih murah menarik lebih banyak orang.

In [ ]:
DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def add_weekday_price(df):
    df = df.copy()
    df["dow"] = df["date_show"].dt.dayofweek
    df["d1_dow"] = df["d1_date"].dt.dayofweek

    def price_on(dates):
        keys = pd.MultiIndex.from_arrays([df["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
        return pd.Series(PRICE_MAP.reindex(keys).values, index=df.index)

    d13_price = pd.concat([price_on(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
    df["price_ratio"] = df["price"] / d13_price
    return df

full = add_weekday_price(full)
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

### Hari libur di sekitar jendela D1-D3 dan hari target
Penjualan D1-D3 terdongkrak bila jatuh pada hari libur, dan hari target bisa berada pada situasi kalender yang berbeda, sehingga model perlu tahu keduanya:
- off_d1, off_d2, off_d3: apakah masing-masing hari pada jendela adalah hari libur.
- win_off, win_hol, win_wk: jumlah hari libur, libur nasional, dan hari akhir pekan pada jendela.
- tgt_off: apakah hari target adalah hari libur.
- off_rel: hari target libur dikurangi porsi hari libur pada jendela, jendela yang penuh hari libur lalu diikuti hari kerja memprediksi penurunan.
- tgt_prev_off, tgt_next_off: apakah hari di samping hari target libur, tepi dari long weekend.
- between_off, between_off_share: jumlah dan porsi hari libur antara D4 dan hari target.

In [ ]:
def is_off(dates):
    return ((dates.dt.dayofweek >= 5) | dates.isin(NATIONAL_HOLIDAYS) | dates.isin(cuti["date"])).astype(int)

def add_window_offdays(df):
    """days off (weekend, national holiday, cuti bersama) in the D1-D3 window, on and around the target day, and between D4 and the target"""
    df = df.copy()
    day = lambda k: df["d1_date"] + pd.Timedelta(days=k)
    for k in range(3):
        df[f"off_d{k + 1}"] = is_off(day(k))
    df["win_off"] = df[["off_d1", "off_d2", "off_d3"]].sum(axis=1)
    df["win_hol"] = sum(day(k).isin(NATIONAL_HOLIDAYS).astype(int) for k in range(3))
    df["win_wk"] = sum((day(k).dt.dayofweek >= 5).astype(int) for k in range(3))
    df["tgt_off"] = is_off(df["date_show"])
    df["off_rel"] = df["tgt_off"] - df["win_off"] / 3
    df["tgt_prev_off"] = is_off(df["date_show"] - pd.Timedelta(days=1))
    df["tgt_next_off"] = is_off(df["date_show"] + pd.Timedelta(days=1))
    df["between_off"] = sum(((df["horizon"] > h) & (is_off(day(h - 1)) == 1)).astype(int) for h in range(4, 11))
    df["between_off_share"] = df["between_off"] / (df["horizon"] - 4).clip(lower=1)
    return df

OFF_COLS = ["off_d1", "off_d2", "off_d3", "win_off", "win_hol", "win_wk", "tgt_off", "off_rel",
            "tgt_prev_off", "tgt_next_off", "between_off", "between_off_share"]
full = add_window_offdays(full)
full.groupby("is_train")[OFF_COLS].mean().round(3)

### Profil film
Jenis film dari external/film_profile.csv, karena penonton film horor lokal, film asing blockbuster, dan sekuel bertahan berbeda:
- is_local: film Indonesia atau asing.
- is_sequel: bagian dari sekuel atau franchise, yang membawa penonton yang sudah ada.
- kind_code: live action, animasi, anime, rilis ulang, konser, atau dokumenter.
- source_code: cerita orisinal, novel, komik, remake, kisah nyata, atau sumber lain.
- origin_code: negara asal.
- predecessor_adm: jumlah penonton film sebelumnya dalam franchise, ukuran penonton yang sudah ada.
- local_horror: lokal dan horor sekaligus, genre yang dibuka kuat dan cepat turun.
Fakta yang baru dipublikasikan setelah 2025-09-30 dikosongkan, sehingga tidak ada informasi setelah cutoff yang masuk ke model.

In [ ]:
PROFILE_CUTOFF = pd.Timestamp("2025-09-30")
FILM_KINDS = ["live_action", "animation", "anime", "rerelease", "concert", "documentary"]
SOURCES = ["original", "novel", "other", "comic", "remake", "true_story", "viral_story", "folklore", "game"]
ORIGINS = ["Indonesia", "United States", "South Korea", "Japan", "Thailand"]

def load_film_profile():
    p = pd.read_csv(f"{EXTERNAL_DIR}/film_profile.csv")
    late = (p["split"] == "test") & ((p["info_public_before_cutoff"] == "no") | (pd.to_datetime(p["fact_source_date"], format="mixed", errors="coerce") > PROFILE_CUTOFF))
    pred_known = pd.to_datetime(p["predecessor_published"], errors="coerce") <= PROFILE_CUTOFF
    first_origin = p["origin_country"].astype(str).str.split(",").str[0].str.strip()
    prof = pd.DataFrame({
        "movie_title": p["movie_title"],
        "is_local": p["is_local"].astype(float),
        "is_sequel": p["is_sequel_or_franchise"].astype(float),
        "kind_code": p["film_kind"].map({k: i for i, k in enumerate(FILM_KINDS)}).astype(float),
        "source_code": p["source_material"].map({k: i for i, k in enumerate(SOURCES)}).astype(float),
        "origin_code": first_origin.map({k: i for i, k in enumerate(ORIGINS)}).fillna(len(ORIGINS)).astype(float),
        "predecessor_adm": np.where(pred_known, np.log1p(p["predecessor_admissions_id"]), np.nan),
    })
    prof.loc[late.values, prof.columns[1:]] = np.nan
    return prof.drop_duplicates("movie_title").set_index("movie_title")

FILM_PROFILE = load_film_profile()
PROFILE_COLS = list(FILM_PROFILE.columns) + ["local_horror"]

def add_film_profile(df):
    df = df.copy()
    for c in FILM_PROFILE.columns:
        df[c] = df["movie_title"].map(FILM_PROFILE[c])
    df["local_horror"] = df["is_local"] * df["genre_Horror"]
    return df

full = add_film_profile(full)
print("films without a profile:", full.loc[full["is_local"].isna()].groupby("is_train")["movie_title"].nunique().to_dict())
full.groupby("is_train")[PROFILE_COLS].mean().round(3)

### Pasangan yang mulai terlambat
Sebuah bioskop bisa menambahkan film setelah D1, sehingga pasangan dengan penjualan hanya pada D3 lebih muda daripada yang ditunjukkan horizonnya.
- first_active: hari pertama pada D1-D3 yang memiliki penjualan.
- eff_age: horizon dikurangi first_active ditambah 1, usia sebenarnya dari pasangan pada hari target.
- last_ratio: penjualan pada hari aktif terakhir relatif terhadap scale, level tempat pasangan mengakhiri D1-D3.
- active_mean: rata-rata penjualan hanya pada hari aktif, karena rata-rata D1-D3 meremehkan pasangan yang mulai terlambat.

In [ ]:
def add_late_start(df):
    df = df.copy()
    t = df[D13].values
    first = np.select([t[:, 0] > 0, t[:, 1] > 0, t[:, 2] > 0], [1, 2, 3], 4)
    last = np.select([t[:, 2] > 0, t[:, 1] > 0], [t[:, 2], t[:, 1]], t[:, 0])
    df["first_active"] = first
    df["eff_age"] = df["horizon"] - first + 1
    df["last_ratio"] = last / df["scale"]
    df["active_mean"] = t.sum(axis=1) / np.clip((t > 0).sum(axis=1), 1, None) / df["scale"]
    return df

full = add_late_start(full)
full.groupby("is_train")["first_active"].value_counts(normalize=True).unstack().round(3)

### Persaingan dari film lain yang baru rilis
Sebuah film kehilangan layar dan penonton ke film yang rilis setelahnya, dan hari pasar yang kuat mengangkat semua film. Fitur ini hanya memakai informasi D1-D3 film lain, tidak pernah targetnya.
- comp_size_cin: ukuran film yang rilis di bioskop yang sama setelah D1 film ini dan sampai hari target, relatif terhadap scale film itu sendiri.
- comp_fresh_cin: sama, tetapi hanya untuk film dalam tiga hari pertamanya, persaingan layar yang paling kuat.
- comp_size_nat: hitungan yang sama pada tingkat nasional.
- date_idx: seberapa kuat pembukaan film lain pada tanggal target dibandingkan bentuk pembukaan yang biasa untuk hari rilisnya, ukuran seberapa ramai pasar hari itu.
- own_idx: ukuran yang sama pada hari D1-D3 film itu sendiri.
- date_idx_rel: date_idx dikurangi own_idx, apakah hari target lebih ramai atau lebih sepi dari hari-hari saat film diamati.

In [ ]:
COMP_MIN_SIZE = 50

def opening_pairs(df):
    return df.drop_duplicates(["movie_title", "cinema_ids"])[["movie_title", "cinema_ids", "d1_date", "t1", "t2", "t3", "scale"]]

def film_openings(df):
    f = opening_pairs(df).groupby(["movie_title", "d1_date"])[D13].sum().reset_index()
    f["m"] = f[D13].mean(axis=1)
    return f[f["m"] >= COMP_MIN_SIZE].reset_index(drop=True)

def typical_opening_shape(df):
    """usual film D_k / mean(D1-D3), k = 1..3, by release weekday"""
    f = film_openings(df)
    log_r = lambda g, k: np.average(np.log((g[f"t{k}"] + 1) / (g["m"] + 1)), weights=np.sqrt(g["m"]))
    overall = [np.exp(log_r(f, k)) for k in (1, 2, 3)]
    by_dow = {dw: [np.exp(log_r(g, k)) for k in (1, 2, 3)] for dw, g in f.groupby(f["d1_date"].dt.dayofweek) if len(g) >= 5}
    return {dw: by_dow.get(dw, overall) for dw in range(7)}

def opening_surprise(df, shape):
    """log of each film's D1-D3 days vs the usual shape for its release weekday, one row per film and date"""
    rows = []
    for r in film_openings(df).itertuples():
        t = shape[r.d1_date.dayofweek]
        for k in range(3):
            rows.append((r.movie_title, r.d1_date + pd.Timedelta(days=k), np.log((getattr(r, f"t{k + 1}") + 1) / (r.m + 1) / t[k]), np.sqrt(r.m)))
    s = pd.DataFrame(rows, columns=["movie_title", "date", "s", "w"])
    return s.assign(sw=s["s"] * s["w"])

def date_strength(dates, titles, surprise):
    """size-weighted mean surprise of the other films in their D1-D3 on each date (NaN when there are none)"""
    tot = surprise.groupby("date")[["sw", "w"]].sum().reindex(pd.DatetimeIndex(dates)).values
    own = surprise.set_index(["date", "movie_title"])[["sw", "w"]].reindex(pd.MultiIndex.from_arrays([pd.DatetimeIndex(dates), titles])).fillna(0).values
    w = tot[:, 1] - own[:, 1]
    ok = w > 1e-9
    return np.where(ok, (tot[:, 0] - own[:, 0]) / np.where(ok, w, 1), np.nan)

def add_competition(df, ref, shape):
    """ref: rows whose pairs are the other films' openings"""
    df = df.reset_index(drop=True)
    others = opening_pairs(ref)
    own_size = opening_pairs(df).groupby("movie_title")["scale"].sum()

    key = df[["movie_title", "cinema_ids", "d1_date", "date_show"]].reset_index()
    m = key.merge(others[["cinema_ids", "movie_title", "d1_date", "scale"]], on="cinema_ids", suffixes=("", "_o"))
    m = m[(m["movie_title_o"] != m["movie_title"]) & (m["d1_date_o"] > m["d1_date"]) & (m["d1_date_o"] <= m["date_show"])]
    fresh = m[(m["date_show"] - m["d1_date_o"]).dt.days <= 2].groupby("index")["scale"].sum()
    df["comp_size_cin"] = np.log1p(m.groupby("index")["scale"].sum().reindex(df.index, fill_value=0).values / df["scale"].values)
    df["comp_fresh_cin"] = np.log1p(fresh.reindex(df.index, fill_value=0).values / df["scale"].values)

    films = others.groupby("movie_title").agg(fd1=("d1_date", "min"), fsize=("scale", "sum")).rename_axis("movie_o").reset_index()
    u = df[["movie_title", "d1_date", "date_show"]].drop_duplicates()
    mm = u.merge(films, how="cross")
    mm = mm[(mm["movie_o"] != mm["movie_title"]) & (mm["fd1"] > mm["d1_date"]) & (mm["fd1"] <= mm["date_show"])]
    u = u.merge(mm.groupby(["movie_title", "d1_date", "date_show"]).agg(comp_size_nat=("fsize", "sum")).reset_index(),
                how="left", on=["movie_title", "d1_date", "date_show"])

    surprise = opening_surprise(ref, shape)
    u["date_idx"] = date_strength(u["date_show"], u["movie_title"].values, surprise)
    own_days = np.vstack([date_strength(u["d1_date"] + pd.Timedelta(days=k), u["movie_title"].values, surprise) for k in range(3)])
    n_days = (~np.isnan(own_days)).sum(axis=0)
    u["own_idx"] = np.where(n_days > 0, np.nansum(own_days, axis=0) / np.maximum(n_days, 1), np.nan)
    u["date_idx_rel"] = u["date_idx"] - u["own_idx"]

    df = df.merge(u, how="left", on=["movie_title", "d1_date", "date_show"])
    df["comp_size_nat"] = np.log1p(df["comp_size_nat"].fillna(0) / df["movie_title"].map(own_size))
    return df

COMP_COLS = ["comp_size_cin", "comp_fresh_cin", "comp_size_nat", "date_idx", "own_idx", "date_idx_rel"]
OPENING_SHAPE = typical_opening_shape(full[full["is_train"] == 1])
is_tr = full["is_train"] == 1
full = pd.concat([add_competition(full[is_tr], full[is_tr], OPENING_SHAPE),
                  add_competition(full[~is_tr], full[~is_tr], OPENING_SHAPE)], ignore_index=True)
full.groupby("is_train")[COMP_COLS].mean().round(3)

### Baris latih tambahan (jendela bergeser)
Hanya ada sekitar dua ratus pembukaan film untuk dipelajari. Menggeser D1 lebih lambat 1 sampai 14 hari memperlakukan hari yang lebih akhir dari film yang sama sebagai D1 dan menghasilkan contoh latih baru dengan tiket nyata, mencakup tahap lain dari masa tayang film dan hari kalender lain.
- age_offset: jumlah hari D1 digeser, 0 untuk baris asli dan baris test, sehingga model dapat membedakan baris hasil geser.
Baris geser hanya dipakai untuk fitting, tidak pernah untuk validasi, dan bobotnya lebih rendah dari baris asli pada model tree karena kurang khas dibanding pembukaan yang sebenarnya.

In [ ]:
AUG_SHIFTS = [1, 2, 3, 5, 7, 10, 14]

def build_shifted_rows(shift):
    d1 = train_d1 + pd.Timedelta(days=shift)
    d1 = d1[d1 + pd.Timedelta(days=9) <= train["date_show"].max()]
    window = build_window(train, d1, 10)
    df = to_long(window[window["t3"] > 0].reset_index(drop=True), with_target=True).assign(is_train=1)
    df = drop_gap_rows(join_external(join_sources(df)))
    df, _ = add_genres(add_calendar_flags(df), GENRE_COLS)
    df = add_target(drop_unused(set_categories(df)))
    df = add_weekday_price(add_cinema_features(add_film_features(add_pair_features(add_off_block(df)), window)))
    df = add_film_profile(add_late_start(add_window_offdays(df)))
    df = add_competition(df, full[full["is_train"] == 1], OPENING_SHAPE)
    return df.assign(age_offset=shift)

full["age_offset"] = 0
AUG = pd.concat([build_shifted_rows(s) for s in AUG_SHIFTS], ignore_index=True)
print(f"extra rows: {len(AUG)} ({len(AUG) / (full['is_train'] == 1).sum():.1f}x the real train rows)")
AUG.groupby("age_offset")["target_ratio"].agg(["size", "mean"]).round(3)

# Validasi Waktu

### Setup
Semua model memprediksi target_ratio, sama seperti notebook utama. Baris train yang menyentuh hari tidak lengkap dihapus, juga dari baris tambahan.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["kumo", "lgbm", "nn"]
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2, "nn": 0.2}
USE_EXTRA_ROWS = True
EXTRA_WEIGHT = 0.5

NUM_FEATURES = [
    "t1", "t2", "t3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
] + OFF_COLS + [
    "first_active", "eff_age", "last_ratio", "active_mean",
] + COMP_COLS + ["age_offset"] + COMMON_GENRES + PROFILE_COLS
FEATURES = NUM_FEATURES + CAT_COLS

def incomplete_days(raw):
    """days on which train.csv is partial (found by a rule): far fewer shows or cinemas than the days around it (21-day median), or no rows at all"""
    g = raw.groupby("date_show").agg(shows=("total_show", "sum"), cinemas=("cinema_ids", "nunique")).reindex(pd.date_range(raw["date_show"].min(), raw["date_show"].max()))
    rel_shows = g["shows"] / g["shows"].rolling(21, center=True, min_periods=7).median()
    rel_cinemas = g["cinemas"] / g["cinemas"].rolling(21, center=True, min_periods=7).median()
    return pd.DatetimeIndex(g.index[(rel_shows < 0.75) | (rel_cinemas < 0.85) | g["shows"].isna()])

INCOMPLETE_DAYS = incomplete_days(train)

def touches_incomplete(df):
    """True for rows whose target day or any of D1-D3 is an incomplete day: their target and their D1-D3 base are under-reported"""
    return (df["date_show"].isin(INCOMPLETE_DAYS) | np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(INCOMPLETE_DAYS) for k in range(3)], axis=0)).values

drop_rows = (full["is_train"] == 1) & touches_incomplete(full)
print(f"incomplete train days: {len(INCOMPLETE_DAYS)} ({INCOMPLETE_DAYS.min().date()} .. {INCOMPLETE_DAYS.max().date()}); train rows dropped: {int(drop_rows.sum())} of {int((full['is_train'] == 1).sum())}")
train_df = full[(full["is_train"] == 1) & ~drop_rows].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
CINEMA_CODE = {c: k for k, c in enumerate(sorted(pd.concat([train_df["cinema_ids"], test_df["cinema_ids"]]).unique()))}
for df_ in (train_df, test_df):
    df_["cinema_code"] = df_["cinema_ids"].map(CINEMA_CODE)
X, y, groups = train_df[FEATURES + ["cinema_code"]], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES + ["cinema_code"]]
AUG = AUG[~touches_incomplete(AUG)].reset_index(drop=True)
AUG["cinema_code"] = AUG["cinema_ids"].map(CINEMA_CODE)
X_aug, y_aug = AUG[FEATURES + ["cinema_code"]], AUG["target_ratio"].values

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def extra_weights(is_extra):
    return np.where(is_extra, EXTRA_WEIGHT, 1.0)

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d, sample_weight=None):
    preds = [make_model(seed).fit(prep(X_tr), y_tr, sample_weight=sample_weight).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

BLEND_SNAP = 0.1
print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | extra rows: {len(AUG)} | test rows: {len(test_df)}")


### Lipatan waktu
Film validasi adalah film dengan D1 di satu bulan, dan data latih hanya film yang jendela D1-D10 nya selesai sedikitnya GAP_DAYS sebelum bulan itu mulai, sehingga model selalu memprediksi masa depan. Baris tambahan memakai aturan yang sama, dan film yang jatuh di jeda tidak dipakai di sisi mana pun.

In [ ]:
TIME_FOLDS = {
    "juli": ("2025-07-01", "2025-07-31"),
    "agustus": ("2025-08-01", "2025-08-31"),
    "september": ("2025-09-01", "2025-09-30"),
}
GAP_DAYS = 10

def split_time_fold(d1_from, d1_to):
    val_mask = train_df["d1_date"].between(pd.Timestamp(d1_from), pd.Timestamp(d1_to)).values
    cut = pd.Timestamp(d1_from) - pd.Timedelta(days=GAP_DAYS)
    ends_before_cut = lambda df: ((df["d1_date"] + pd.Timedelta(days=9)) < cut).values
    return val_mask, ends_before_cut(train_df), ends_before_cut(AUG), cut

summary = []
for name, (d1_from, d1_to) in TIME_FOLDS.items():
    val_mask, tr_mask, aug_mask, cut = split_time_fold(d1_from, d1_to)
    summary.append({"lipatan": name, "film validasi": train_df.loc[val_mask, "movie_title"].nunique(), "baris validasi": int(val_mask.sum()),
                    "baris latih asli": int(tr_mask.sum()), "baris latih tambahan": int(aug_mask.sum()),
                    "D10 film latih sebelum": str(cut.date()),
                    "mean target validasi": round(float(y[val_mask].mean()), 3), "mean target latih": round(float(y[tr_mask].mean()), 3)})
pd.DataFrame(summary)

# Regressor Models

### LGBM Regressor
Gradient boosted trees dengan objective absolute error yang sesuai dengan metrik. Baris tambahan diberi bobot sampel lebih rendah dari baris asli, dan prediksi beberapa seed dirata-rata agar hasil tidak bergantung pada satu seed yang kebetulan bagus.

In [ ]:
LGBM_PARAMS = dict(
    n_estimators=1500,
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    make_model = lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, sample_weight=extra_weights(is_extra))


### Siapkan fitur untuk model non-tree
Model linear, SVM, dan logistic regression belajar dari angka mentah, sehingga fitur numerik dipotong di persentil 1 dan 99, nilai kosong diisi median, lalu distandarisasi. Kolom kategori diubah menjadi kolom 0 atau 1 (one-hot) dengan level yang sama pada data latih dan validasi.

In [ ]:
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

def linear_prep(X_tr, X_pred):
    lo, hi, med = X_tr[NUM_COLS].quantile(0.01), X_tr[NUM_COLS].quantile(0.99), X_tr[NUM_COLS].median()
    num = lambda d: d[NUM_COLS].astype(float).clip(lo, hi, axis=1).fillna(med)
    ref = num(X_tr)
    mu, sd = ref.mean(), ref.std() + 1e-9
    encode = lambda d: np.hstack([((num(d) - mu) / sd).values, pd.get_dummies(d[CAT_COLS], dtype=float).values]).astype(np.float32)
    return encode(X_tr[FEATURES]), encode(X_pred[FEATURES])

A_demo, B_demo = linear_prep(X.iloc[:2000], X.iloc[2000:3000])
print(A_demo.shape, B_demo.shape)

### Linear Regression (ElasticNet)
Regresi linear dengan penalti L1 dan L2. Lossnya kuadrat dan bukan absolut, sehingga tidak sesuai dengan metrik, dan hubungannya garis lurus sehingga tidak menangkap interaksi.

In [ ]:
def linreg_fit_predict(X_tr, y_tr, is_extra, X_pred):
    A, B = linear_prep(X_tr, X_pred)
    model = ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=2000, random_state=SEED).fit(A, y_tr, sample_weight=extra_weights(is_extra))
    return model.predict(B)

### SVM Regressor
Support vector regression dengan kernel linear dan loss epsilon-insensitive dengan epsilon 0, yaitu loss absolut seperti metrik. Kernel RBF tidak dipakai karena biayanya naik kuadrat terhadap jumlah baris.

In [ ]:
def svm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    A, B = linear_prep(X_tr, X_pred)
    model = LinearSVR(loss="epsilon_insensitive", epsilon=0.0, C=1.0, dual="auto", max_iter=3000, random_state=SEED).fit(A, y_tr, sample_weight=extra_weights(is_extra))
    return model.predict(B)

### Logistic Regression
Logistic regression adalah model klasifikasi, sehingga dipakai untuk menebak apakah tiket hari target sama dengan nol atau tidak. Hasilnya dipakai di dua cara, yaitu skor klasifikasi nol dibanding LGBM, dan dua tahap di mana prediksi Linear Regression dijadikan 0 bila peluang tidak nol di bawah 0,5.

In [ ]:
def logreg_fit_predict(X_tr, y_tr, is_extra, X_pred):
    A, B = linear_prep(X_tr, X_pred)
    model = LogisticRegression(C=1.0, max_iter=300).fit(A, (y_tr > 0).astype(int), sample_weight=extra_weights(is_extra))
    return model.predict_proba(B)[:, 1]

### LSTM
Tiga hari pertama dibaca sebagai deret pendek (tiket dan show tiap hari dibagi scale pasangan itu), lalu keadaan akhir LSTM digabung dengan fitur lain dan embedding kategori untuk memprediksi target_ratio dengan loss absolut. Deretnya hanya tiga langkah sehingga LSTM hanya punya sedikit pola waktu untuk dipelajari.

In [ ]:
LSTM_EPOCHS, LSTM_HIDDEN, LSTM_BATCH, LSTM_LR, LSTM_SEEDS = 6, 32, 512, 2e-3, [1]
SEQ_T, SEQ_S = ["t1", "t2", "t3"], ["show1", "show2", "show3"]
STATIC_NUM = [c for c in NUM_COLS if c not in SEQ_T + SEQ_S]

class SeqNet(torch.nn.Module):
    def __init__(self, n_static, cardinalities):
        super().__init__()
        self.emb = torch.nn.ModuleList([torch.nn.Embedding(c, min(8, (c + 1) // 2 + 1)) for c in cardinalities])
        self.lstm = torch.nn.LSTM(input_size=2, hidden_size=LSTM_HIDDEN, batch_first=True)
        dim = LSTM_HIDDEN + n_static + sum(e.embedding_dim for e in self.emb)
        self.head = torch.nn.Sequential(torch.nn.Linear(dim, 128), torch.nn.SiLU(), torch.nn.Dropout(0.2), torch.nn.Linear(128, 64), torch.nn.SiLU(), torch.nn.Linear(64, 1))

    def forward(self, seq, static, cat):
        _, (h, _) = self.lstm(seq)
        return self.head(torch.cat([h[-1], static] + [e(cat[:, k]) for k, e in enumerate(self.emb)], dim=1)).squeeze(1)

def lstm_tensors(d, stats):
    seq = np.stack([(d[SEQ_T].values / d[["scale"]].values), (d[SEQ_S].values / (d[["show_mean"]].values + 1e-6))], axis=2).clip(0, 10)
    static = ((d[STATIC_NUM].astype(float).clip(stats["lo"], stats["hi"], axis=1).fillna(stats["med"]) - stats["mu"]) / stats["sd"]).values
    cat = np.stack([d[c].cat.codes.values + 1 for c in CAT_COLS] + [d["cinema_code"].values + 1], axis=1)
    return torch.tensor(seq, dtype=torch.float32), torch.tensor(static, dtype=torch.float32), torch.tensor(cat, dtype=torch.int64)

def lstm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    ref = X_tr[STATIC_NUM].astype(float)
    lo, hi, med = ref.quantile(0.01), ref.quantile(0.99), ref.median()
    clipped = ref.clip(lo, hi, axis=1).fillna(med)
    stats = {"lo": lo, "hi": hi, "med": med, "mu": clipped.mean(), "sd": clipped.std() + 1e-9}
    cards = [len(X_tr[c].cat.categories) + 2 for c in CAT_COLS] + [len(CINEMA_CODE) + 2]
    seq, static, cat = lstm_tensors(X_tr, stats)
    seq_p, static_p, cat_p = lstm_tensors(X_pred, stats)
    target = torch.tensor(y_tr, dtype=torch.float32)
    weight = torch.tensor(extra_weights(is_extra), dtype=torch.float32)
    preds = []
    for seed in LSTM_SEEDS:
        torch.manual_seed(seed)
        net = SeqNet(static.shape[1], cards)
        opt = torch.optim.AdamW(net.parameters(), lr=LSTM_LR, weight_decay=1e-5)
        for epoch in range(LSTM_EPOCHS):
            net.train()
            order = torch.randperm(len(seq))
            for start in range(0, len(seq), LSTM_BATCH):
                idx = order[start:start + LSTM_BATCH]
                if len(idx) < 2:
                    continue
                loss = (weight[idx] * (net(seq[idx], static[idx], cat[idx]) - target[idx]).abs()).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            preds.append(net(seq_p, static_p, cat_p).numpy())
    return np.mean(preds, axis=0)

### Latih dan prediksi per lipatan
Setiap model dilatih sekali per lipatan di CPU. Prediksi di bawah ZERO_SNAP dijadikan 0 seperti pada notebook utama, dan logistic regression menyimpan peluang tidak nol.

In [ ]:
MODEL_SEEDS = [42]
ZERO_SNAP.update({"linreg": 0.2, "svm": 0.2, "lstm": 0.2})
REGRESSORS = {"lgbm": lgbm_fit_predict, "linreg": linreg_fit_predict, "svm": svm_fit_predict, "lstm": lstm_fit_predict}

def run_fold(name):
    val_mask, tr_mask, aug_mask, _ = split_time_fold(*TIME_FOLDS[name])
    X_fit = pd.concat([X[tr_mask], X_aug[aug_mask]], ignore_index=True)
    y_fit = np.r_[y[tr_mask], y_aug[aug_mask]]
    is_extra = np.r_[np.zeros(int(tr_mask.sum()), bool), np.ones(int(aug_mask.sum()), bool)]
    X_val = X[val_mask]
    preds = {}
    for m, fit_predict in REGRESSORS.items():
        start = time.time()
        p = np.clip(fit_predict(X_fit, y_fit, is_extra, X_val), 0, None)
        preds[m] = np.where(p < ZERO_SNAP.get(m, 0.0), 0.0, p)
        print(f"{name} {m}: MASE={mase_ratio(y[val_mask], preds[m]):.4f} ({time.time() - start:.0f} s)")
    prob = logreg_fit_predict(X_fit, y_fit, is_extra, X_val)
    preds["logreg + linreg (dua tahap)"] = np.where(prob >= 0.5, preds["linreg"], 0.0)
    print(f"{name} dua tahap: MASE={mase_ratio(y[val_mask], preds['logreg + linreg (dua tahap)']):.4f}")
    return {"y": y[val_mask], "film": groups[val_mask].values, "preds": preds, "prob": prob}

RESULTS = {name: run_fold(name) for name in TIME_FOLDS}

# Perbandingan Model

### Skor tiap model
Skor model yang baru dijalankan, ditambah skor Kumo, NN, dan blend dari blend_poc_time_cv.ipynb yang memakai lipatan, baris, dan fitur yang sama. Kumo dan NN butuh GPU sehingga tidak dijalankan ulang di sini, dan LGBM di sana memakai 3 seed. Model akhir yang dipresentasikan adalah Kumo 0,7 dan LGBM 0,3.

In [ ]:
REFERENCE = pd.DataFrame(
    {"juli": [0.3520, 0.3824, 0.3471, 0.3480], "agustus": [0.3218, 0.4591, 0.3229, 0.3191], "september": [0.2400, 0.2539, 0.2537, 0.2407],
     "rata-rata lipatan": [0.3046, 0.3651, 0.3079, 0.3026], "gabungan": [0.3129, 0.3771, 0.3148, 0.3105]},
    index=["kumo (blend_poc_time_cv)", "nn MLP (blend_poc_time_cv)", "lgbm 3 seed (blend_poc_time_cv)", "kumo 0.7 / lgbm 0.3 (blend_poc_time_cv)"],
)
ALL_Y = np.concatenate([r["y"] for r in RESULTS.values()])
MODELS = list(next(iter(RESULTS.values()))["preds"])
rows = {"naif rasio 1 (D1-D3)": {n: mase_ratio(r["y"], 1.0) for n, r in RESULTS.items()}, "naif 0": {n: mase_ratio(r["y"], 0.0) for n, r in RESULTS.items()}}
for m in MODELS:
    rows[m] = {n: mase_ratio(r["y"], r["preds"][m]) for n, r in RESULTS.items()}
comparison = pd.DataFrame(rows).T
comparison["rata-rata lipatan"] = comparison[list(RESULTS)].mean(axis=1)
comparison["gabungan"] = [mase_ratio(ALL_Y, 1.0), mase_ratio(ALL_Y, 0.0)] + [mase_ratio(ALL_Y, np.concatenate([r["preds"][m] for r in RESULTS.values()])) for m in MODELS]
pd.concat([comparison, REFERENCE]).round(4)

### Menebak baris target nol
Logistic regression dibandingkan dengan LGBM sebagai penebak apakah target sama dengan nol. LGBM dianggap menebak tidak nol bila prediksinya di atas 0 setelah zero-snap, dan acuan sederhananya adalah menebak semua baris tidak nol.

In [ ]:
y_nonzero = (ALL_Y > 0).astype(int)
prob = np.concatenate([r["prob"] for r in RESULTS.values()])
lgbm_nonzero = (np.concatenate([r["preds"]["lgbm"] for r in RESULTS.values()]) > 0).astype(int)
pd.DataFrame({
    "akurasi": [y_nonzero.mean(), accuracy_score(y_nonzero, prob >= 0.5), accuracy_score(y_nonzero, lgbm_nonzero)],
    "F1 (tidak nol)": [f1_score(y_nonzero, np.ones_like(y_nonzero)), f1_score(y_nonzero, prob >= 0.5), f1_score(y_nonzero, lgbm_nonzero)],
    "AUC": [0.5, roc_auc_score(y_nonzero, prob), np.nan],
}, index=["tebak semua tidak nol", "logistic regression", "lgbm (prediksi > 0)"]).round(4)

### Korelasi error antar model
Error bertanda (prediksi dikurangi aktual) pada seluruh baris validasi gabungan. Korelasi yang tinggi berarti model-model itu membuat kesalahan yang hampir sama.

In [ ]:
err = pd.DataFrame({m: np.concatenate([r["preds"][m] for r in RESULTS.values()]) - ALL_Y for m in MODELS})
err.corr().round(3)

### Bootstrap per film terhadap LGBM
Film validasi diambil ulang dengan pengembalian sebanyak 2000 kali, lalu selisih MASE tiap model terhadap LGBM dihitung pada sampel yang sama. Selisih negatif berarti model itu lebih baik dari LGBM.

In [ ]:
ALL_FILM = np.concatenate([r["film"] for r in RESULTS.values()])
films = pd.factorize(ALL_FILM)[0]
n_films = films.max() + 1
counts = np.bincount(films, minlength=n_films)
film_error = lambda p: np.bincount(films, weights=np.abs(ALL_Y - p), minlength=n_films)
pooled = {m: np.concatenate([r["preds"][m] for r in RESULTS.values()]) for m in MODELS}
draws = np.random.default_rng(SEED).integers(0, n_films, size=(2000, n_films))
rows = []
for m in MODELS:
    if m == "lgbm":
        continue
    diff = film_error(pooled[m]) - film_error(pooled["lgbm"])
    boot = diff[draws].sum(axis=1) / counts[draws].sum(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rows.append({"model": m, "film": n_films, "selisih MASE vs lgbm": round(diff.sum() / counts.sum(), 4), "CI 95%": f"[{lo:.4f}, {hi:.4f}]", "P(lebih baik dari lgbm)": round(float((boot < 0).mean()), 3)})
pd.DataFrame(rows)